# LeetCode 649 — Dota2 Senate bằng Python

Link đề: https://leetcode.com/problems/dota2-senate/ · Độ khó: **Medium** · Chủ đề: Greedy, Queue, String

Skeleton LeetCode:

```python
class Solution:
    def predictPartyVictory(self, senate: str) -> str:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với một bộ **tìm kiếm trò chơi vét cạn** độc lập trên mọi chuỗi độ dài ≤ 9 và so giữa các cách giải), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai phe `R` (Radiant) và `D` (Dire) ngồi thành một hàng; đi lần lượt từ đầu đến cuối theo **vòng**, mỗi thượng nghị sĩ còn quyền **cấm quyền** một người khác (người đó mất quyền ở mọi vòng sau), hoặc **tuyên bố thắng** nếu chỉ còn một phe. Hãy dự đoán phe thắng khi **mọi người chơi tối ưu**.
2. Nắm các công cụ Python: `collections.deque` (`append`, `popleft`), `enumerate` để lấy chỉ số, `str.count`, biểu thức điều kiện `a if cond else b`, nhãn "thời gian" `chỉ số + n·vòng`, và `itertools.product` để thử mọi chuỗi nhỏ.
3. Đi từ **mô phỏng thẳng bằng mảng cờ còn sống (`O(n²)`)** → **hai hàng đợi chỉ số với mẹo `i + n` (`O(n)`)** → **một hàng đợi ký tự với bộ đếm "nợ cấm" (`O(n)`)**; đối chiếu với **tìm kiếm trò chơi vét cạn (minimax)** để chắc chắn chiến thuật tham lam đúng.
4. Hiểu mấu chốt: chiến thuật tối ưu là **cấm đối thủ sẽ hành động sớm nhất tiếp theo** (người đối địch gần nhất phía sau mình, theo vòng tròn). Vì mỗi lượt cấm đúng một người, ta chỉ cần so **ai đứng trước** giữa hai đầu hàng đợi của hai phe.
5. Tránh các lỗi kinh điển: đoán theo **số đông**, quên đưa người thắng **trở lại cuối hàng đợi**, quên cộng `n` (thời gian vòng sau), và cấm **sai người** (không phải người đối địch gần nhất).

## 2. Đề bài & ví dụ

Trong thế giới Dota2, có hai phe: **Radiant** và **Dire**. Thượng viện Dota2 gồm các thượng nghị sĩ của hai phe. Họ muốn quyết định một thay đổi trong game bằng một thủ tục bỏ phiếu **theo vòng**. Trong mỗi vòng, mỗi thượng nghị sĩ có thể thực hiện **một** trong hai quyền:

- **Cấm quyền một thượng nghị sĩ:** làm một thượng nghị sĩ khác **mất toàn bộ quyền** ở vòng này và **mọi vòng sau**.
- **Tuyên bố chiến thắng:** nếu thượng nghị sĩ nhận thấy những người **còn quyền** đều thuộc **cùng một phe**, người đó có thể tuyên bố thắng và quyết định thay đổi.

Cho chuỗi `senate` mô tả phe của từng thượng nghị sĩ: `'R'` là Radiant, `'D'` là Dire. Nếu có `n` thượng nghị sĩ thì độ dài chuỗi là `n`.

Thủ tục chạy **từ thượng nghị sĩ đầu tiên đến cuối cùng theo thứ tự cho trước** (vòng 1), rồi lại từ đầu (vòng 2), ... cho đến khi kết thúc. Những người đã mất quyền sẽ **bị bỏ qua**.

Giả sử mọi người đều đủ thông minh và chơi chiến thuật tốt nhất cho phe mình. Hãy dự đoán phe nào cuối cùng **tuyên bố thắng**. Kết quả là `"Radiant"` hoặc `"Dire"`.

**Ví dụ 1**

- Input: `senate = "RD"`
- Output: `"Radiant"`
- Giải thích: người thứ nhất (Radiant) cấm quyền người thứ hai ngay vòng 1. Người thứ hai không còn quyền. Vòng 2, người thứ nhất là người duy nhất còn quyền nên tuyên bố thắng.

**Ví dụ 2**

- Input: `senate = "RDD"`
- Output: `"Dire"`
- Giải thích: `R` đầu cấm `D` thứ hai. `D` thứ ba (Dire) cấm `R` đầu. Vòng 2, `D` thứ ba là người duy nhất còn quyền nên tuyên bố thắng.

**Ràng buộc:**

- `n == senate.length`
- `1 <= n <= 10^4`
- `senate[i]` là `'R'` hoặc `'D'`.

Chú ý: người đã hành động ở vòng này **vẫn còn quyền** ở vòng sau (trừ khi bị cấm). Một người bị cấm **trước khi tới lượt** sẽ mất luôn lượt của mình.

In [1]:
from typing import List, Optional
from collections import deque
from functools import lru_cache
import itertools
import random

examples = [
    ("RD", "Radiant"),
    ("RDD", "Dire"),
]
extra_examples = [  # tự dựng, đã kiểm chứng bằng code; nên đối chiếu lại đề gốc
    ("R", "Radiant"),
    ("RRDDD", "Radiant"),
    ("DDR", "Dire"),
]
for s, expected in examples + extra_examples:
    print(repr(s), "-> kỳ vọng", expected)

'RD' -> kỳ vọng Radiant
'RDD' -> kỳ vọng Dire
'R' -> kỳ vọng Radiant
'RRDDD' -> kỳ vọng Radiant
'DDR' -> kỳ vọng Dire


## 3. Kiến thức nền

### 3.1. `collections.deque`: hàng đợi hai đầu

`deque` (đọc "đếch") là hàng đợi: `append(x)` thêm vào **cuối**, `popleft()` lấy ra **đầu**, đều tốn `O(1)`. Truy cập đầu hàng đợi là `q[0]`. Đừng dùng `list.pop(0)` thay `popleft()`: lấy phần tử đầu của `list` phải **dịch cả danh sách** nên tốn `O(n)`.

In [2]:
q = deque([3, 5, 8])
print(q[0])          # 3  (đầu hàng)
q.append(9)          # thêm vào cuối
x = q.popleft()      # lấy đầu ra
print(x, list(q))    # 3 [5, 8, 9]
print(bool(deque()), bool(q))   # False True  (hàng rỗng là "giả")

3
3 [5, 8, 9]
False True


### 3.2. `enumerate` và việc tách chỉ số theo phe

`enumerate(s)` cho cặp `(chỉ số, ký tự)`. Ta dùng nó để chia các chỉ số vào hai danh sách: chỉ số của `R` và chỉ số của `D`. Chỉ số cho biết **ai đứng trước ai**, tức là **ai hành động trước**.

In [3]:
senate = "RRDDD"
qr = deque()
qd = deque()
for i, c in enumerate(senate):
    if c == "R":
        qr.append(i)
    else:
        qd.append(i)
print("R:", list(qr), "D:", list(qd))     # R: [0, 1] D: [2, 3, 4]

R: [0, 1] D: [2, 3, 4]


### 3.3. Nhãn "thời gian" `chỉ số + n·(số vòng)`

Người ở vị trí `i` hành động ở "thời điểm" `i` trong vòng 1, `i + n` trong vòng 2, `i + 2n` trong vòng 3, ... (với `n` là số người). Nếu ta cho mỗi người một **nhãn thời gian** như thế thì "ai hành động trước" chỉ là **so sánh hai số nguyên**. Sau khi hành động ở thời điểm `t`, người đó sẽ hành động tiếp ở `t + n`.

In [4]:
n = 5
t = 1                       # người ở vị trí 1, vòng 1
for rnd in range(1, 4):
    print(f"vòng {rnd}: thời điểm {t}  (vị trí {t % n}, vòng {t // n + 1})")
    t += n

vòng 1: thời điểm 1  (vị trí 1, vòng 1)
vòng 2: thời điểm 6  (vị trí 1, vòng 2)
vòng 3: thời điểm 11  (vị trí 1, vòng 3)


### 3.4. `str.count` và biểu thức điều kiện

`senate.count("R")` đếm số ký tự `R` trong chuỗi. `"Radiant" if x else "Dire"` chọn một trong hai chuỗi (biểu thức điều kiện). Ta dùng để trả đáp án từ việc "hàng đợi nào còn người".

In [5]:
senate = "RRDDD"
print(senate.count("R"), senate.count("D"))       # 2 3
qr = deque([5])
print("Radiant" if qr else "Dire")                  # Radiant
print("Radiant" if deque() else "Dire")             # Dire

2 3
Radiant
Dire


### 3.5. `itertools.product`: mọi chuỗi `R`/`D` có độ dài `k`

Để kiểm chứng đáp án ta sẽ thử **mọi** chuỗi nhỏ. `itertools.product("RD", repeat=k)` sinh `2^k` bộ ký tự.

In [6]:
print(["".join(t) for t in itertools.product("RD", repeat=2)])      # ['RR', 'RD', 'DR', 'DD']
print(sum(1 for _ in itertools.product("RD", repeat=10)))          # 1024

['RR', 'RD', 'DR', 'DD']
1024


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Câu hỏi quan trọng nhất:** khi đến lượt mình, nên cấm **ai**?

Hãy xét `"RDD"`. `R` ở vị trí 0 đến lượt trước tiên. Nếu `R` **không** cấm `D` nào, thì `D` ở vị trí 1 sẽ cấm `R`, và `R` mất ngay. Vì vậy `R` bắt buộc phải xuống tay trước: cấm một `D`. Nhưng cấm `D` nào? Có hai `D`:

- Cấm `D` ở vị trí 1: còn `D` ở vị trí 2 sẽ hành động tiếp và cấm `R` ⇒ **Dire thắng**.
- Cấm `D` ở vị trí 2: `D` ở vị trí 1 vẫn còn, sẽ hành động tiếp và cũng cấm `R` ⇒ **Dire thắng**.

Ở ví dụ này hai lựa chọn cùng kết quả. Nhưng ở `"RRDDD"` (hai `R`, ba `D`) thì khác. Ta thử chiến thuật: **cấm đối thủ hành động sớm nhất sau mình** (người đối địch gần nhất phía sau, vòng tròn):

| Lượt (vòng 1) | Người hành động | Cấm | Còn lại |
|---|---|---|---|
| 1 | `R` vị trí 0 | `D` vị trí 2 | `R0 R1 D3 D4` |
| 2 | `R` vị trí 1 | `D` vị trí 3 | `R0 R1 D4` |
| 3 | `D` vị trí 4 | `R` vị trí 0 (người đối địch gần nhất kế tiếp theo vòng tròn) | `R1 D4` |
| (vòng 2) 4 | `R` vị trí 1 | `D` vị trí 4 | `R1` |

Kết quả **Radiant** thắng dù chỉ có 2 người so với 3 người! Vậy **không thể đoán theo số đông**.

**Vì sao cấm "người đối địch gần nhất kế tiếp"?** Người đó là kẻ **sắp hành động** và sẽ cấm một người của mình. Cấm hắn sớm nhất là cách giảm tổn thất nhiều nhất. Còn cấm một người xa (hành động muộn) thì kẻ gần kia vẫn kịp ra tay. Điều này sẽ được kiểm chứng bằng vét cạn ở mục 12.

**Hệ quả quan trọng:** vì mỗi người hành động đều cấm **đúng một** đối thủ, ta có thể ghép đôi: "người hành động sớm nhất của một phe" ↔ "người hành động sớm nhất của phe kia". Ai đứng trước thì **thắng cặp đó** (cấm người kia), rồi phải **chờ vòng sau**. Đó là nền tảng của lời giải hàng đợi.

In [7]:
# Kiểm tra tay bảng trên bằng danh sách
alive = {0: "R", 1: "R", 3: "D", 4: "D"}     # sau lượt 1 (D2 bị cấm)
del alive[3]                                  # lượt 2: R1 cấm D3
print(alive)                                  # {0: 'R', 1: 'R', 4: 'D'}
del alive[0]                                  # lượt 3: D4 cấm R0
print(alive)                                  # {1: 'R', 4: 'D'}
del alive[4]                                  # vòng 2, R1 cấm D4
print(alive)                                  # {1: 'R'}  -> Radiant thắng

{0: 'R', 1: 'R', 4: 'D'}
{1: 'R', 4: 'D'}
{1: 'R'}


## 5. Cách 1 — Brute Force / baseline: mô phỏng thẳng bằng mảng cờ còn sống

**Ý tưởng.** Làm đúng theo đề:

1. `alive = [True] * n`: ai còn quyền.
2. Lặp các vòng: `for i in range(n)`; nếu `alive[i]` thì người `i` hành động.
3. Khi hành động, tìm **đối thủ gần nhất phía sau** (theo vòng tròn): thử `j = (i + 1) % n, (i + 2) % n, ...`, lấy người đầu tiên **còn sống và khác phe**.
4. Nếu **không** tìm thấy đối thủ nào ⇒ chỉ còn một phe ⇒ người `i` tuyên bố thắng ⇒ trả về phe của `i`.
5. Ngược lại cấm đối thủ: `alive[j] = False`.

In [8]:
def senate_bruteforce_demo(senate):
    n = len(senate)
    alive = [True] * n
    rnd = 0
    while True:
        rnd += 1
        for i in range(n):
            if not alive[i]:
                continue
            target = -1
            for step in range(1, n):
                j = (i + step) % n
                if alive[j] and senate[j] != senate[i]:
                    target = j
                    break
            if target == -1:
                print(f"vòng {rnd}: người {i} ({senate[i]}) không còn đối thủ -> tuyên bố thắng")
                return "Radiant" if senate[i] == "R" else "Dire"
            alive[target] = False
            print(f"vòng {rnd}: người {i} ({senate[i]}) cấm người {target} ({senate[target]})")


print(senate_bruteforce_demo("RRDDD"))

vòng 1: người 0 (R) cấm người 2 (D)
vòng 1: người 1 (R) cấm người 3 (D)
vòng 1: người 4 (D) cấm người 0 (R)
vòng 2: người 1 (R) cấm người 4 (D)
vòng 3: người 1 (R) không còn đối thủ -> tuyên bố thắng
Radiant


In [9]:
def senate_bruteforce(senate):
    """Cách 1: mô phỏng bằng mảng cờ, tìm đối thủ gần nhất phía sau. Time O(n^2)."""
    n = len(senate)
    alive = [True] * n
    while True:
        for i in range(n):
            if not alive[i]:
                continue
            target = -1
            for step in range(1, n):
                j = (i + step) % n
                if alive[j] and senate[j] != senate[i]:
                    target = j
                    break
            if target == -1:
                return "Radiant" if senate[i] == "R" else "Dire"
            alive[target] = False


for s, expected in examples + extra_examples:
    got = senate_bruteforce(s)
    print(repr(s), got, got == expected)

'RD' Radiant True
'RDD' Dire True
'R' Radiant True
'RRDDD' Radiant True
'DDR' Dire True


**Độ phức tạp.** Mỗi lần hành động tìm đối thủ mất tới `O(n)` bước, và có đúng `n - 1` lần cấm (mỗi lần cấm loại một người, đến khi còn một phe), nên Time **`O(n²)`** (với `n = 10^4` là cỡ `10^8` bước tệ nhất, quá chậm trong Python); ngoài ra ta còn duyệt qua cả những người đã bị cấm ở mỗi vòng. Space `O(n)`.

**Nút thắt.** Mỗi lần ta **tìm lại từ đầu** đối thủ gần nhất, trong khi ta đã biết "hàng" của mỗi phe theo thứ tự. Nếu mỗi phe giữ **một hàng đợi** các người của mình (theo thứ tự hành động), thì đối thủ gần nhất chính là **đầu hàng đợi của phe kia**: lấy trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Ba nhận xét:

1. **Chiến thuật tối ưu là cấm đối thủ gần nhất kế tiếp** (mục 4, được chứng minh bằng lập luận trao đổi ở mục 10 và kiểm chứng vét cạn ở mục 12).
2. **Hai hàng đợi chỉ số.** Gọi `qr` là hàng đợi chỉ số các người `R` (đã sắp tăng), `qd` cho `D`. Lấy đầu mỗi hàng: `r = qr[0]`, `d = qd[0]`. **Ai nhỏ hơn thì hành động trước** và cấm người kia. Người thắng cặp phải chờ vòng sau ⇒ đưa về **cuối** hàng đợi của phe mình với nhãn thời gian `chỉ số + n`. Người bị cấm thì không được đưa lại. Hàng đợi nào cạn trước thì phe đó thua.
3. **Một hàng đợi + bộ đếm nợ.** Thay vì ghép cặp tường minh ta có thể duyệt một hàng đợi các ký tự, nhớ "phe `D` còn **nợ** bao nhiêu lệnh cấm sẽ giáng xuống các `D` sắp được duyệt" (`ban_d`) và tương tự `ban_r`. Khi gặp một `D` mà `ban_d > 0` thì `D` đó đã bị cấm (trừ một nợ, loại khỏi hàng); nếu không thì `D` ra tay: tăng `ban_r` (một `R` sẽ bị cấm) và quay lại cuối hàng.

## 7. Các cách tiếp theo

### Cách 2 — Hai hàng đợi chỉ số, nhãn thời gian `i + n` (cách chuẩn)

**Giải thích từng đoạn.**

- `qr`, `qd`: hàng đợi chỉ số của `R` và `D`.
- `while qr and qd:` còn cả hai phe thì còn cặp để đấu.
- `r, d = qr[0], qd[0]`: hai người sẽ hành động sớm nhất của mỗi phe.
- Nếu `r < d`: `R` hành động trước, cấm `D`. `R` được đưa lại cuối hàng với nhãn `r + n` (vòng sau). Ngược lại thì `D` cấm `R` và quay lại với `d + n`.
- Cả hai đầu hàng đều được lấy ra (`popleft`): một người bị cấm, một người thắng nhưng đã được đưa lại ở cuối.
- Hết vòng lặp: phe nào còn người thì thắng.

In [10]:
def senate_queue_demo(senate):
    n = len(senate)
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    print("ban đầu: qr =", list(qr), " qd =", list(qd))
    step = 0
    while qr and qd:
        step += 1
        r, d = qr[0], qd[0]
        if r < d:
            print(f"bước {step}: r={r} < d={d} -> R thắng cặp, cấm D; R quay lại với nhãn {r + n}")
            qr.append(r + n)
        else:
            print(f"bước {step}: d={d} < r={r} -> D thắng cặp, cấm R; D quay lại với nhãn {d + n}")
            qd.append(d + n)
        qr.popleft()
        qd.popleft()
        print("         qr =", list(qr), " qd =", list(qd))
    return "Radiant" if qr else "Dire"


print(senate_queue_demo("RRDDD"))

ban đầu: qr = [0, 1]  qd = [2, 3, 4]
bước 1: r=0 < d=2 -> R thắng cặp, cấm D; R quay lại với nhãn 5
         qr = [1, 5]  qd = [3, 4]
bước 2: r=1 < d=3 -> R thắng cặp, cấm D; R quay lại với nhãn 6
         qr = [5, 6]  qd = [4]
bước 3: d=4 < r=5 -> D thắng cặp, cấm R; D quay lại với nhãn 9
         qr = [6]  qd = [9]
bước 4: r=6 < d=9 -> R thắng cặp, cấm D; R quay lại với nhãn 11
         qr = [11]  qd = []
Radiant


In [11]:
class Solution2:
    def predictPartyVictory(self, senate: str) -> str:
        qr = deque()
        qd = deque()
        for i, c in enumerate(senate):
            if c == "R":
                qr.append(i)
            else:
                qd.append(i)
        n = len(senate)
        while qr and qd:
            if qr[0] < qd[0]:
                qr.append(qr[0] + n)
            else:
                qd.append(qd[0] + n)
            qr.popleft()
            qd.popleft()
        return "Radiant" if qr else "Dire"


for s, expected in examples + extra_examples:
    assert Solution2().predictPartyVictory(s) == expected
print("Cách 2 đúng trên các ví dụ")

Cách 2 đúng trên các ví dụ


**Độ phức tạp Cách 2:** mỗi lần lặp **loại đúng một người** (người bị cấm không bao giờ quay lại), nên có tối đa `n - 1` lần lặp, mỗi lần `O(1)` ⇒ Time **`O(n)`**. Space `O(n)` cho hai hàng đợi.

---

### Cách 3 — Một hàng đợi ký tự và bộ đếm "nợ cấm"

**Ý tưởng.** Không cần chỉ số. Duyệt một hàng đợi `q` chứa các ký tự (thứ tự = thứ tự hành động), cùng hai số đếm còn sống `cr`, `cd` và hai bộ đếm nợ:

- `ban_r`: số `R` **sắp bị cấm** (do `D` đã ra tay) nhưng chưa tới lượt bị duyệt tới.
- `ban_d`: số `D` sắp bị cấm.

Khi lấy một `R` ra khỏi hàng:

- Nếu `ban_r > 0`: `R` này **đã bị cấm** (một `D` đứng trước đã chọn hắn là "đối thủ gần nhất kế tiếp"): `ban_r -= 1`, `cr -= 1`, **không** đưa lại.
- Ngược lại `R` này hành động: tăng `ban_d += 1` (một `D` sẽ bị cấm, đúng là `D` kế tiếp được duyệt tới) và **đưa `R` về cuối hàng** để vòng sau còn tham gia.

`D` đối xứng. Dừng khi `cr == 0` hoặc `cd == 0`.

Vì sao "nợ" gán đúng người? Người bị cấm bởi `R` hành động là **`D` đầu tiên được duyệt tới sau đó** trong thứ tự hàng đợi vòng tròn: đó chính là "đối thủ gần nhất kế tiếp".

In [12]:
def senate_debt_demo(senate):
    q = deque(senate)
    cr = senate.count("R")
    cd = len(senate) - cr
    ban_r = ban_d = 0
    step = 0
    while cr and cd:
        step += 1
        c = q.popleft()
        if c == "R":
            if ban_r:
                ban_r -= 1
                cr -= 1
                what = "R này đã bị cấm -> bỏ"
            else:
                ban_d += 1
                q.append("R")
                what = "R hành động: nợ cấm D +1, quay lại cuối hàng"
        else:
            if ban_d:
                ban_d -= 1
                cd -= 1
                what = "D này đã bị cấm -> bỏ"
            else:
                ban_r += 1
                q.append("D")
                what = "D hành động: nợ cấm R +1, quay lại cuối hàng"
        print(f"{step:>2}. lấy {c}: {what:<46} ban_r={ban_r} ban_d={ban_d} cr={cr} cd={cd} hàng={''.join(q)}")
    return "Radiant" if cr else "Dire"


print(senate_debt_demo("RRDDD"))

 1. lấy R: R hành động: nợ cấm D +1, quay lại cuối hàng   ban_r=0 ban_d=1 cr=2 cd=3 hàng=RDDDR
 2. lấy R: R hành động: nợ cấm D +1, quay lại cuối hàng   ban_r=0 ban_d=2 cr=2 cd=3 hàng=DDDRR
 3. lấy D: D này đã bị cấm -> bỏ                          ban_r=0 ban_d=1 cr=2 cd=2 hàng=DDRR
 4. lấy D: D này đã bị cấm -> bỏ                          ban_r=0 ban_d=0 cr=2 cd=1 hàng=DRR
 5. lấy D: D hành động: nợ cấm R +1, quay lại cuối hàng   ban_r=1 ban_d=0 cr=2 cd=1 hàng=RRD
 6. lấy R: R này đã bị cấm -> bỏ                          ban_r=0 ban_d=0 cr=1 cd=1 hàng=RD
 7. lấy R: R hành động: nợ cấm D +1, quay lại cuối hàng   ban_r=0 ban_d=1 cr=1 cd=1 hàng=DR
 8. lấy D: D này đã bị cấm -> bỏ                          ban_r=0 ban_d=0 cr=1 cd=0 hàng=R
Radiant


In [13]:
class Solution3:
    def predictPartyVictory(self, senate: str) -> str:
        q = deque(senate)
        cr = senate.count("R")
        cd = len(senate) - cr
        ban_r = ban_d = 0
        while cr and cd:
            c = q.popleft()
            if c == "R":
                if ban_r:
                    ban_r -= 1
                    cr -= 1
                else:
                    ban_d += 1
                    q.append("R")
            else:
                if ban_d:
                    ban_d -= 1
                    cd -= 1
                else:
                    ban_r += 1
                    q.append("D")
        return "Radiant" if cr else "Dire"


for s, expected in examples + extra_examples:
    assert Solution3().predictPartyVictory(s) == expected
print("Cách 3 đúng trên các ví dụ")

Cách 3 đúng trên các ví dụ


**Độ phức tạp Cách 3:** mỗi lần lặp hoặc **loại một người** hoặc **đưa một người quay lại**; số lần "quay lại" mỗi vòng bằng số người còn trụ và sau mỗi vòng quân số giảm khoảng một nửa (mỗi người hành động loại một người), nên tổng bước là `O(n)` (tổng cấp số nhân `n + n/2 + n/4 + ...`). Ở mục 12 ta đo thực nghiệm tỉ lệ số lần lấy ra / `n`. Space `O(n)`.

Cách 2 và Cách 3 cùng là "mô phỏng bằng hàng đợi", nhưng khác thật ở **cách ghi nhớ**: Cách 2 ghép cặp tường minh bằng nhãn thời gian; Cách 3 chỉ giữ hai **bộ đếm** và để thứ tự hàng đợi tự quyết định ai bị cấm.

## 8. TRACE từng bước

**Bảng trạng thái cho Cách 2 với `senate = "RRDDD"` (`n = 5`):**

| Bước | `qr` | `qd` | `r`, `d` | Ai thắng cặp | Hành động | `qr` sau | `qd` sau |
|---|---|---|---|---|---|---|---|
| 0 | `[0, 1]` | `[2, 3, 4]` | | | khởi tạo | | |
| 1 | `[0, 1]` | `[2, 3, 4]` | `0`, `2` | `R` (`0 < 2`) | `R0` cấm `D2`; `R0` quay lại nhãn `5` | `[1, 5]` | `[3, 4]` |
| 2 | `[1, 5]` | `[3, 4]` | `1`, `3` | `R` | `R1` cấm `D3`; quay lại nhãn `6` | `[5, 6]` | `[4]` |
| 3 | `[5, 6]` | `[4]` | `5`, `4` | `D` (`4 < 5`) | `D4` cấm `R0` (nhãn `5`); `D4` quay lại nhãn `9` | `[6]` | `[9]` |
| 4 | `[6]` | `[9]` | `6`, `9` | `R` | `R1` (nhãn `6`) cấm `D4` (nhãn `9`); quay lại nhãn `11` | `[11]` | `[]` |

`qd` rỗng ⇒ **Radiant** thắng. Nhãn `11 = 1 + 2·5` nghĩa là người sống sót là `R` ở vị trí `1`, sẽ hành động ở **vòng 3**.

Hàm `trace_queue` dưới đây in lại đúng bảng đó (có kèm **nhật ký cấm**: ai cấm ai, theo vị trí gốc `nhãn % n`).

In [14]:
def trace_queue(senate):
    n = len(senate)
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    log = []
    print("khởi tạo: qr =", list(qr), "qd =", list(qd))
    step = 0
    while qr and qd:
        step += 1
        r, d = qr[0], qd[0]
        if r < d:
            winner, loser = ("R", r), ("D", d)
            qr.append(r + n)
        else:
            winner, loser = ("D", d), ("R", r)
            qd.append(d + n)
        qr.popleft()
        qd.popleft()
        log.append((winner[1] % n, loser[1] % n))
        print(f"bước {step}: {winner[0]}{winner[1] % n} (nhãn {winner[1]}) cấm "
              f"{loser[0]}{loser[1] % n} (nhãn {loser[1]}) | qr = {list(qr)} qd = {list(qd)}")
    res = "Radiant" if qr else "Dire"
    print("kết quả:", res, "| nhật ký cấm (người cấm, người bị cấm):", log)
    return res


assert trace_queue("RRDDD") == "Radiant"

khởi tạo: qr = [0, 1] qd = [2, 3, 4]
bước 1: R0 (nhãn 0) cấm D2 (nhãn 2) | qr = [1, 5] qd = [3, 4]
bước 2: R1 (nhãn 1) cấm D3 (nhãn 3) | qr = [5, 6] qd = [4]
bước 3: D4 (nhãn 4) cấm R0 (nhãn 5) | qr = [6] qd = [9]
bước 4: R1 (nhãn 6) cấm D4 (nhãn 9) | qr = [11] qd = []
kết quả: Radiant | nhật ký cấm (người cấm, người bị cấm): [(0, 2), (1, 3), (4, 0), (1, 4)]


**Trace thêm cho `"RDD"` (ví dụ 2), `n = 3`:** `qr = [0]`, `qd = [1, 2]`. Bước 1: `0 < 1`, `R0` cấm `D1`, `R0` quay lại nhãn `3`: `qr = [3]`, `qd = [2]`. Bước 2: `r = 3`, `d = 2`: `d < r`, `D2` cấm `R0` (nhãn 3), `D2` quay lại nhãn `5`: `qr = []`, `qd = [5]`. `qr` rỗng ⇒ **Dire**.

In [15]:
assert trace_queue("RDD") == "Dire"

khởi tạo: qr = [0] qd = [1, 2]
bước 1: R0 (nhãn 0) cấm D1 (nhãn 1) | qr = [3] qd = [2]
bước 2: D2 (nhãn 2) cấm R0 (nhãn 3) | qr = [] qd = [5]
kết quả: Dire | nhật ký cấm (người cấm, người bị cấm): [(0, 1), (2, 0)]


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (hai hàng đợi chỉ số, nhãn `i + n`): `O(n)` và là lời giải chuẩn. Trên LeetCode `deque` thường đã được import sẵn; để chắc chắn khi nộp, thêm `from collections import deque` ở đầu (trong notebook đã import ở trên).

In [16]:
class Solution:
    def predictPartyVictory(self, senate: str) -> str:
        qr = deque()
        qd = deque()
        for i, c in enumerate(senate):
            if c == "R":
                qr.append(i)
            else:
                qd.append(i)
        n = len(senate)
        while qr and qd:
            if qr[0] < qd[0]:
                qr.append(qr[0] + n)      # R thắng cặp, sang vòng sau
            else:
                qd.append(qd[0] + n)      # D thắng cặp, sang vòng sau
            qr.popleft()
            qd.popleft()
        return "Radiant" if qr else "Dire"


for s, expected in examples + extra_examples:
    assert Solution().predictPartyVictory(s) == expected
print("Code nộp đúng trên 5 ví dụ")

Code nộp đúng trên 5 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Chiến thuật tham lam: cấm đối thủ gần nhất kế tiếp là tối ưu.** Xét lượt của người `x` (phe `P`); gọi `y` là đối thủ sẽ hành động **sớm nhất sau `x`**. Lập luận trao đổi (trực giác, không phải chứng minh hình thức đầy đủ): giả sử `x` cấm một đối thủ khác `z` thay vì `y`. Khi đó `y` còn sống và sẽ ra tay cấm một người của `P` ở một thời điểm **sớm hơn** mọi lượt của `z`. Nếu `x` cấm `y` thay vì `z`, thì phe đối thủ vẫn mất đúng **một** người, nhưng những lượt cấm mà phe đó thực hiện từ nay về sau chỉ còn các thời điểm **muộn hơn hoặc bằng** (mất đi thời điểm sớm nhất của `y`, còn `z` vốn đã muộn). Lượt cấm của đối thủ càng muộn thì phe `P` càng có nhiều lượt ra tay trước nên càng có lợi. Cấm đồng đội thì đương nhiên không có lợi. Vì vậy cấm `y` không bao giờ tệ hơn các lựa chọn khác. Để khỏi phải tin vào lập luận, mục 12 **kiểm chứng bằng vét cạn trò chơi** (minimax, được phép cấm bất kỳ ai, kể cả đồng đội) cho mọi chuỗi độ dài ≤ 9: luôn trùng với chiến thuật này.

**(b) Hai hàng đợi mô phỏng đúng.** Bất biến: `qr` (và `qd`) chứa **nhãn thời gian tăng dần** của các người còn quyền của phe đó, mỗi người đúng một nhãn là thời điểm hành động **kế tiếp** của họ. Người hành động sớm nhất toàn cục là phần tử nhỏ hơn trong `{qr[0], qd[0]}`. Giả sử đó là `r = qr[0] < d = qd[0]`. `R` này cấm đối thủ gần nhất kế tiếp: đó là người `D` có nhãn nhỏ nhất **lớn hơn `r`**, tức là `d = qd[0]` (vì `qd[0]` là nhỏ nhất trong `qd` và `qd[0] > r`). Cập nhật: `d` bị loại (popleft), `r` hành động xong nên nhãn kế tiếp là `r + n` (xếp cuối `qr` vì mọi nhãn khác trong `qr` đều `< r + n`: các nhãn trong `qr` luôn nằm trong một khoảng ít hơn `n` tính từ nhãn nhỏ nhất). Bất biến được giữ. Trường hợp `d < r` đối xứng.

**(c) Dừng và kết luận.** Mỗi lần lặp loại đúng một người nên sau tối đa `n - 1` lần lặp một hàng đợi cạn. Khi đó chỉ còn một phe có quyền: người sống sót kế tiếp sẽ tuyên bố thắng cho phe đó. Hàng đợi không rỗng ⇒ phe đó thắng.

**(d) Vì sao Cách 3 (bộ đếm nợ) tương đương.** Duyệt vòng tròn theo thứ tự hành động; mỗi lần một người hành động, "đối thủ gần nhất kế tiếp" chính là đối thủ đầu tiên mà con trỏ duyệt tới sau đó; ghi nợ `ban_*` rồi trả nợ khi duyệt tới đối thủ đó **mô phỏng đúng** việc cấm đó. Mục 12 kiểm chứng nhật ký cấm của Cách 1 và Cách 2 **trùng khớp từng cặp** trên mọi chuỗi độ dài ≤ 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mảng cờ + tìm đối thủ | `O(n²)` | `O(n)` | chậm với `n = 10^4` |
| 2. Hai hàng đợi chỉ số (nộp) | `O(n)` | `O(n)` | ≤ `n-1` lần lặp, mỗi lần `O(1)` |
| 3. Một hàng đợi + bộ đếm nợ | `O(n)` (khấu hao) | `O(n)` | số vòng ~ `log n`, mỗi vòng giảm ~ một nửa |

Chú ý: **cận dưới** `Ω(n)` vì phải đọc mọi ký tự. Vì thế `O(n)` là tối ưu. Với `n ≤ 10^4` cả Cách 2 và 3 chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ đề, biên, và oracle vét cạn (minimax)

Oracle độc lập **không dùng chiến thuật tham lam**: `senate_minimax` thử **mọi** lựa chọn người bị cấm (kể cả cấm đồng đội) và để mỗi phe chọn nước đi tốt nhất cho mình (trò chơi hai người dạng minimax; mỗi lượt hành động hoặc cấm một người). Trạng thái là `(tập người còn sống, người đang hành động)`; dùng `lru_cache` nên với `n ≤ 9` rất nhanh.

In [17]:
def senate_minimax(senate):
    """Oracle vét cạn: mọi người chọn mọi khả năng cấm, mỗi phe tối ưu cho mình."""
    n = len(senate)

    @lru_cache(maxsize=None)
    def go(alive, pos):
        # alive: bitmask những người còn quyền; pos: người đang hành động (còn sống)
        has_r = any(alive >> i & 1 and senate[i] == "R" for i in range(n))
        has_d = any(alive >> i & 1 and senate[i] == "D" for i in range(n))
        if not has_d:
            return "R"
        if not has_r:
            return "D"
        me = senate[pos]
        other = "D" if me == "R" else "R"
        for j in range(n):
            if j == pos or not (alive >> j & 1):
                continue
            nxt_alive = alive & ~(1 << j)
            nxt = pos
            for k in range(1, n + 1):          # người còn sống kế tiếp theo vòng tròn
                p = (pos + k) % n
                if nxt_alive >> p & 1:
                    nxt = p
                    break
            if go(nxt_alive, nxt) == me:
                return me
        return other

    return "Radiant" if go((1 << n) - 1, 0) == "R" else "Dire"


for s, expected in examples + extra_examples:
    assert senate_minimax(s) == expected, s
print("Oracle minimax khớp 5 ví dụ")

Oracle minimax khớp 5 ví dụ


In [18]:
solvers = {
    "Cách 1 mảng cờ": senate_bruteforce,
    "Cách 2 hai hàng đợi": Solution2().predictPartyVictory,
    "Cách 3 bộ đếm nợ": Solution3().predictPartyVictory,
    "Code nộp": Solution().predictPartyVictory,
}

# Ví dụ đề + tự dựng + biên
edge = [("R", "Radiant"), ("D", "Dire"), ("RD", "Radiant"), ("DR", "Dire"),
        ("RR", "Radiant"), ("DD", "Dire"), ("RDR", "Radiant"), ("DRD", "Dire")]
for s, expected in examples + extra_examples + edge:
    for name, f in solvers.items():
        assert f(s) == expected, (name, s)
print("Ví dụ và biên đều đúng với mọi cách")

Ví dụ và biên đều đúng với mọi cách


### 12.2. Vét cạn toàn bộ chuỗi nhỏ (so với oracle minimax)

Mọi chuỗi độ dài `1..9` (tổng `1022` chuỗi) so với `senate_minimax`.

In [19]:
total = 0
for k in range(1, 10):
    for t in itertools.product("RD", repeat=k):
        s = "".join(t)
        truth = senate_minimax(s)
        for name, f in solvers.items():
            assert f(s) == truth, (name, s, truth)
        total += 1
print("Đã so", total, "chuỗi độ dài 1..9 với oracle minimax: cả 4 cách đều khớp")

Đã so 1022 chuỗi độ dài 1..9 với oracle minimax: cả 4 cách đều khớp


### 12.3. So các cách với nhau trên mọi chuỗi độ dài ≤ 12, và so nhật ký cấm

Nhật ký cấm: danh sách `(vị trí người cấm, vị trí người bị cấm)` theo thứ tự thời gian. Cách 1 và Cách 2 phải sinh **cùng một nhật ký**, đó là kiểm tra mạnh hơn nhiều so với chỉ so đáp án.

In [20]:
def ban_log_bruteforce(senate):
    n = len(senate)
    alive = [True] * n
    log = []
    while True:
        for i in range(n):
            if not alive[i]:
                continue
            target = -1
            for step in range(1, n):
                j = (i + step) % n
                if alive[j] and senate[j] != senate[i]:
                    target = j
                    break
            if target == -1:
                return log
            alive[target] = False
            log.append((i, target))


def ban_log_queue(senate):
    n = len(senate)
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    log = []
    while qr and qd:
        r, d = qr[0], qd[0]
        if r < d:
            qr.append(r + n)
            log.append((r % n, d % n))
        else:
            qd.append(d + n)
            log.append((d % n, r % n))
        qr.popleft()
        qd.popleft()
    return log


count = 0
for k in range(1, 13):
    for t in itertools.product("RD", repeat=k):
        s = "".join(t)
        ref = senate_bruteforce(s)
        assert Solution2().predictPartyVictory(s) == ref == Solution3().predictPartyVictory(s)
        assert ban_log_bruteforce(s) == ban_log_queue(s), s
        count += 1
print("Đã so", count, "chuỗi độ dài 1..12; nhật ký cấm của Cách 1 và Cách 2 trùng khớp")

Đã so 8190 chuỗi độ dài 1..12; nhật ký cấm của Cách 1 và Cách 2 trùng khớp


### 12.4. Ngẫu nhiên có seed, đúng ràng buộc và cỡ lớn

Chuỗi ngẫu nhiên độ dài tới `10^4` (đủ cỡ ràng buộc) với các tỉ lệ `R` khác nhau, cộng các dạng cực đoan. Cách 1 chỉ chạy trên cỡ nhỏ (chậm). Ta cũng đo **số lần lấy ra / n** của Cách 3 để minh họa `O(n)` khấu hao.

In [21]:
rng = random.Random(649)
for _ in range(400):
    n = rng.randint(1, 60)
    p = rng.choice([0.1, 0.3, 0.5, 0.7, 0.9])
    s = "".join("R" if rng.random() < p else "D" for _ in range(n))
    ref = senate_bruteforce(s)
    assert Solution2().predictPartyVictory(s) == ref == Solution3().predictPartyVictory(s)

big = []
for n in (1, 2, 10 ** 3, 10 ** 4):
    for p in (0.01, 0.5, 0.99):
        big.append("".join("R" if rng.random() < p else "D" for _ in range(n)))
big += ["R" * 5000 + "D" * 5000, "D" * 5000 + "R" * 5000, "RD" * 5000, "DR" * 5000,
        "R" * 9999 + "D", "D" * 9999 + "R", "R" * 10 ** 4, "D" * 10 ** 4]
for s in big:
    assert len(s) <= 10 ** 4
    assert Solution2().predictPartyVictory(s) == Solution3().predictPartyVictory(s)
print("Ngẫu nhiên và cỡ lớn: Cách 2 và 3 khớp trên", len(big), "chuỗi lớn")


def debt_pops(senate):
    q = deque(senate)
    cr = senate.count("R")
    cd = len(senate) - cr
    ban_r = ban_d = pops = 0
    while cr and cd:
        c = q.popleft()
        pops += 1
        if c == "R":
            if ban_r:
                ban_r -= 1
                cr -= 1
            else:
                ban_d += 1
                q.append("R")
        else:
            if ban_d:
                ban_d -= 1
                cd -= 1
            else:
                ban_r += 1
                q.append("D")
    return pops


worst = max(debt_pops(s) / len(s) for s in big)
print("tỉ lệ số lần lấy ra / n lớn nhất trên các chuỗi lớn:", round(worst, 2))
assert worst < 4

Ngẫu nhiên và cỡ lớn: Cách 2 và 3 khớp trên 20 chuỗi lớn
tỉ lệ số lần lấy ra / n lớn nhất trên các chuỗi lớn: 1.77


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đoán theo số đông

Nhiều người nghĩ "phe đông hơn thắng". Nhưng **thứ tự** quyết định: một người ra tay sớm sẽ loại đối thủ trước khi họ kịp ra tay.

In [22]:
def sai_so_dong(senate):
    return "Radiant" if senate.count("R") > senate.count("D") else "Dire"


s = "RRDDD"
print("sai :", sai_so_dong(s))                         # Dire
print("đúng:", Solution().predictPartyVictory(s))      # Radiant
assert sai_so_dong(s) != Solution().predictPartyVictory(s)

sai : Dire
đúng: Radiant


**Vì sao sai:** `RRDDD` có 2 `R`, 3 `D` nhưng hai `R` đứng đầu hành động trước và cấm hai `D`; sau đó chỉ còn một `D` đấu một `R` và `R` hành động trước ở vòng sau. **Sửa:** mô phỏng đúng thứ tự (hai hàng đợi).

### Lỗi 2: Quên đưa người thắng về cuối hàng đợi

Người thắng cặp **vẫn còn quyền** ở vòng sau; nếu ta chỉ `popleft` mà không `append` thì người đó biến mất như bị cấm.

In [23]:
def sai_khong_dua_lai(senate):
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    while qr and qd:
        qr.popleft()                    # SAI: cả hai bị lấy ra, không ai quay lại
        qd.popleft()
    return "Radiant" if qr else "Dire"


s = "RRDDD"
print("sai :", sai_khong_dua_lai(s))                   # Dire (chỉ so số lượng!)
print("đúng:", Solution().predictPartyVictory(s))
assert sai_khong_dua_lai(s) != Solution().predictPartyVictory(s)

sai : Dire
đúng: Radiant


**Vì sao sai:** mỗi cặp loại **cả hai** người, nên kết quả chỉ phụ thuộc vào ai đông hơn, quay về lỗi 1. **Sửa:** thêm `qr.append(qr[0] + n)` hoặc `qd.append(qd[0] + n)` **trước** khi `popleft`.

### Lỗi 3: Đưa lại nhưng quên cộng `n`

Nếu đưa lại cùng chỉ số, người thắng sẽ luôn "đứng trước" những ai ở vòng trước (nhãn nhỏ) và thắng mãi.

In [24]:
def sai_khong_cong_n(senate):
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    while qr and qd:
        if qr[0] < qd[0]:
            qr.append(qr[0])            # SAI: phải là qr[0] + n
        else:
            qd.append(qd[0])
        qr.popleft()
        qd.popleft()
    return "Radiant" if qr else "Dire"


print("sai :", sai_khong_cong_n("RDD"))                # Radiant
print("đúng:", Solution().predictPartyVictory("RDD"))  # Dire
assert sai_khong_cong_n("RDD") != "Dire"

sai : Radiant
đúng: Dire


**Vì sao sai:** ở `RDD`, `R0` thắng cặp đầu rồi quay lại với nhãn `0` (đáng lẽ `3`), nên so với `D2` (nhãn `2`) vẫn "đứng trước" và lại thắng, trong khi thực tế `D2` ở vòng 1 **đã hành động trước** `R0` của vòng 2. **Sửa:** nhãn vòng sau là `chỉ số + n`.

### Lỗi 4: Cấm sai người (không phải đối thủ gần nhất kế tiếp)

Ví dụ cấm **đối thủ xa nhất** (người cuối hàng của phe kia) hay cấm "đối thủ đầu hàng" bất kể vòng. Ta thử cấm đối thủ còn sống có **chỉ số lớn nhất** và tìm chuỗi đầu tiên cho kết quả khác:

In [25]:
def sai_cam_xa_nhat(senate):
    n = len(senate)
    alive = [True] * n
    while True:
        for i in range(n):
            if not alive[i]:
                continue
            opp = [j for j in range(n) if alive[j] and senate[j] != senate[i]]
            if not opp:
                return "Radiant" if senate[i] == "R" else "Dire"
            alive[max(opp)] = False         # SAI: cấm người có chỉ số lớn nhất


found = None
for k in range(1, 8):
    for t in itertools.product("RD", repeat=k):
        s = "".join(t)
        if sai_cam_xa_nhat(s) != Solution().predictPartyVictory(s):
            found = s
            break
    if found:
        break
print("chuỗi ngắn nhất cho kết quả sai:", found)
print("sai :", sai_cam_xa_nhat(found), "| đúng:", Solution().predictPartyVictory(found), "| minimax:", senate_minimax(found))
assert sai_cam_xa_nhat(found) != senate_minimax(found)

chuỗi ngắn nhất cho kết quả sai: RDRDD
sai : Dire | đúng: Radiant | minimax: Radiant


**Vì sao sai:** người đối địch **hành động sớm nhất** vẫn còn đó và sẽ ra tay cấm người của mình. **Sửa:** luôn cấm đối thủ đầu hàng đợi phe kia (đối thủ gần nhất kế tiếp).

### Lỗi 5: Dùng `list.pop(0)` thay `deque.popleft()`

Về đúng/sai thì không khác, nhưng `list.pop(0)` dịch cả danh sách nên mỗi lần `O(n)`; với `n = 10^4` Cách 2 sẽ chuyển từ `O(n)` thành `O(n²)` trong trường hợp xấu. Dùng `deque` để giữ `O(1)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mỗi người lần lượt, vòng sau lại từ đầu", "mất quyền", "người chơi tối ưu", hai phe đối kháng ⇒ **mô phỏng bằng hàng đợi** + **tham lam** (loại đối thủ sắp ra tay nhất).
- **Chuyển "vòng" thành "nhãn thời gian"** `i + n·vòng`: so sánh ai trước chỉ là so sánh hai số.
- **Mỗi lần xử lý loại đúng một phần tử** ⇒ số bước ≤ `n` ⇒ `O(n)`; luôn tìm bất biến kiểu này khi phân tích mô phỏng.
- **Tránh đoán theo số đông / công thức**: hãy thử `RRDDD` (thiểu số vẫn thắng nhờ đứng trước).
- **Kiểm chứng tham lam bằng vét cạn** trên chuỗi nhỏ (`itertools.product`): kỹ thuật rất hữu ích khi chưa chắc chiến thuật tối ưu.
- **Hai hàng đợi** là mẫu hay gặp ở bài đối kháng hai phe theo lượt (ví dụ Reveal Cards In Increasing Order, Card game, Josephus mở rộng).
- **Bộ đếm nợ** (`pending`) thay cho việc xóa phần tử ngay: lazy deletion.

## 15. Biến thể

1. **Trả về nhật ký cấm / số người sống sót:** dùng lại Cách 2 và ghi `(người cấm, người bị cấm)` hay `len` của hàng đợi còn lại.
2. **Có nhiều hơn hai phe:** chiến thuật "cấm đối thủ kế tiếp" không còn rõ ràng (phải chọn phe nào để cấm), bài trở thành bài lý thuyết trò chơi khó hơn.
3. **Mỗi người có số lần cấm khác nhau / có "trọng số":** hàng đợi ưu tiên theo nhãn thời gian thay vì hai hàng đợi.
4. **Hỏi vòng nào tuyên bố thắng:** dựa vào nhãn của người sống sót (`nhãn // n + 1`) nhưng cần xử lý cẩn thận trường hợp người sống sót hành động ngay trong vòng của lần cấm cuối.
5. **Josephus problem:** loại người theo bước nhảy cố định, cũng mô phỏng bằng hàng đợi/công thức.

In [26]:
def survivors_and_log(senate):
    """Biến thể 1: trả (phe thắng, danh sách vị trí gốc còn sống, nhật ký cấm)."""
    n = len(senate)
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    log = []
    while qr and qd:
        r, d = qr[0], qd[0]
        if r < d:
            qr.append(r + n)
            log.append((r % n, d % n))
        else:
            qd.append(d + n)
            log.append((d % n, r % n))
        qr.popleft()
        qd.popleft()
    winner_q = qr if qr else qd
    return ("Radiant" if qr else "Dire"), sorted(x % n for x in winner_q), log


print(survivors_and_log("RRDDD"))     # ('Radiant', [1], [(0, 2), (1, 3), (4, 0), (1, 4)])
print(survivors_and_log("DDDRR"))     # phe D thắng, xem danh sách sống sót và nhật ký

('Radiant', [1], [(0, 2), (1, 3), (4, 0), (1, 4)])
('Dire', [0, 1, 2], [(0, 3), (1, 4)])


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng cờ + tìm đối thủ | Mô phỏng đúng luật, tìm đối thủ gần nhất kế tiếp mỗi lượt | `O(n²)` | `O(n)` | Làm oracle/hiểu đề; không nộp |
| 2. Hai hàng đợi chỉ số | Mỗi phe một hàng đợi; đầu hàng nhỏ hơn thắng cặp, quay lại với `+n` | `O(n)` | `O(n)` | **Nộp bài** (chuẩn, dễ chứng minh) |
| 3. Một hàng đợi + bộ đếm nợ | Duyệt vòng tròn; ghi nợ cấm cho phe kia, trả nợ khi gặp | `O(n)` khấu hao | `O(n)` | Khi muốn gọn, không cần chỉ số |
| (Oracle) Minimax | Thử mọi nước đi, mỗi phe tối ưu | mũ, chỉ `n ≤ 9` | mũ | Chỉ để kiểm chứng tham lam |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().predictPartyVictory("DDR")`; `Solution().predictPartyVictory("R")`; `list(deque([1, 2, 3]).popleft() for _ in range(1))`; `7 % 5` và `7 // 5` (liên hệ nhãn `7` khi `n = 5`).

**Bài 2 (tính tay).** Với `senate = "DRDRR"`, lập bảng như mục 8 (hai hàng đợi `qr`, `qd`, từng bước, ai cấm ai) và cho biết phe thắng.

**Bài 3 (điền TODO).** Hoàn thành `senate_todo(senate)` theo Cách 2.

In [27]:
def senate_todo(senate):
    # TODO 1: tạo hai deque qr, qd chứa chỉ số của 'R' và 'D'; n = len(senate)
    # TODO 2: while qr and qd: so qr[0] với qd[0]; bên nhỏ hơn thắng, append(chỉ số + n) vào hàng của bên đó
    # TODO 3: popleft cả hai hàng
    # TODO 4: return "Radiant" nếu qr còn, ngược lại "Dire"
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `senate_debt(senate)` theo Cách 3 (một hàng đợi ký tự, hai bộ đếm nợ `ban_r`, `ban_d` và hai số đếm còn sống).

**Bài 5 (biến thể / mini challenge).** (a) Viết `winner_survivors(senate)` trả về **số người sống sót** của phe thắng (khi kết thúc). (b) Viết `first_ban_pairs(senate, k)` trả về `k` lượt cấm đầu tiên dưới dạng `(vị trí người cấm, vị trí người bị cấm)`; kiểm chứng bằng `ban_log_bruteforce`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"Dire"` (`D0` cấm `R2`, `D1` vẫn còn); `"Radiant"`; `1`; `2` và `1` (nhãn `7` khi `n = 5` là vị trí `2` ở vòng `2`). Bài 3: bám sát mục 9.
- **Mức 2 (vừa):** Bài 2: `qr = [1, 3, 4]`, `qd = [0, 2]`. Bước 1: `d = 0 < r = 1` ⇒ `D0` cấm `R1`, `D0` quay lại nhãn `5`. Bước 2: `r = 3`, `d = 2` ⇒ `D2` cấm `R3`, quay lại nhãn `7`. Bước 3: `r = 4`, `d = 5` ⇒ `R4` cấm `D0` (nhãn `5`), `R4` quay lại nhãn `9`. Bước 4: `r = 9`, `d = 7` ⇒ `D2` (nhãn `7`) cấm `R4`; `qr` rỗng ⇒ **Dire**. Bài 4: nhớ `q.append` lại chính ký tự vừa hành động.
- **Mức 3 (sâu):** Bài 5(a): khi vòng lặp Cách 2 kết thúc, `len(qr or qd)` là số người sống sót của phe thắng (vì mỗi người bị loại ngay lúc bị cấm). Bài 5(b): ghi `log.append((r % n, d % n))` khi `r < d`, ngược lại `(d % n, r % n)`; dừng khi `len(log) == k` hoặc một hàng cạn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert Solution().predictPartyVictory("DDR") == "Dire"
assert Solution().predictPartyVictory("R") == "Radiant"
assert list(deque([1, 2, 3]).popleft() for _ in range(1)) == [1]
assert 7 % 5 == 2 and 7 // 5 == 1

# Bài 2
assert trace_queue("DRDRR") == "Dire"
assert ban_log_queue("DRDRR") == [(0, 1), (2, 3), (4, 0), (2, 4)]


# Bài 3
def senate_todo_ans(senate):
    n = len(senate)
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    while qr and qd:
        if qr[0] < qd[0]:
            qr.append(qr[0] + n)
        else:
            qd.append(qd[0] + n)
        qr.popleft()
        qd.popleft()
    return "Radiant" if qr else "Dire"


# Bài 4
def senate_debt(senate):
    q = deque(senate)
    cr = senate.count("R")
    cd = len(senate) - cr
    ban_r = ban_d = 0
    while cr and cd:
        c = q.popleft()
        if c == "R":
            if ban_r:
                ban_r -= 1
                cr -= 1
            else:
                ban_d += 1
                q.append("R")
        else:
            if ban_d:
                ban_d -= 1
                cd -= 1
            else:
                ban_r += 1
                q.append("D")
    return "Radiant" if cr else "Dire"


# Bài 5
def winner_survivors(senate):
    n = len(senate)
    qr = deque(i for i, c in enumerate(senate) if c == "R")
    qd = deque(i for i, c in enumerate(senate) if c == "D")
    while qr and qd:
        if qr[0] < qd[0]:
            qr.append(qr[0] + n)
        else:
            qd.append(qd[0] + n)
        qr.popleft()
        qd.popleft()
    return len(qr or qd)


def first_ban_pairs(senate, k):
    return ban_log_queue(senate)[:k]


rng3 = random.Random(6490)
for _ in range(300):
    n = rng3.randint(1, 40)
    s = "".join(rng3.choice("RD") for _ in range(n))
    ref = senate_bruteforce(s)
    assert senate_todo_ans(s) == senate_debt(s) == ref
    log = ban_log_bruteforce(s)
    assert winner_survivors(s) == n - len(log)
    k = rng3.randint(0, 5)
    assert first_ban_pairs(s, k) == log[:k]
assert winner_survivors("RRDDD") == 1
assert first_ban_pairs("RRDDD", 2) == [(0, 2), (1, 3)]
print("Tất cả đáp án bài tập đều đúng.")

khởi tạo: qr = [1, 3, 4] qd = [0, 2]
bước 1: D0 (nhãn 0) cấm R1 (nhãn 1) | qr = [3, 4] qd = [2, 5]
bước 2: D2 (nhãn 2) cấm R3 (nhãn 3) | qr = [4] qd = [5, 7]
bước 3: R4 (nhãn 4) cấm D0 (nhãn 5) | qr = [9] qd = [7]
bước 4: D2 (nhãn 7) cấm R4 (nhãn 9) | qr = [] qd = [12]
kết quả: Dire | nhật ký cấm (người cấm, người bị cấm): [(0, 1), (2, 3), (4, 0), (2, 4)]
Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai hàng đợi chỉ số, nhãn thời gian `i + n`)

```python
class Solution:
    def predictPartyVictory(self, senate: str) -> str:
        qr, qd = deque(), deque()
        for i, c in enumerate(senate):
            (qr if c == "R" else qd).append(i)     # chỉ số = thời điểm hành động vòng 1
        n = len(senate)
        while qr and qd:
            if qr[0] < qd[0]:
                qr.append(qr[0] + n)               # R thắng cặp, sang vòng sau
            else:
                qd.append(qd[0] + n)               # D thắng cặp, sang vòng sau
            qr.popleft()                           # một người bị cấm, một người đã đưa lại
            qd.popleft()
        return "Radiant" if qr else "Dire"
```

**Checklist**

- [ ] Có **hai** hàng đợi chỉ số (mỗi phe một hàng), sắp theo thứ tự hành động?
- [ ] So `qr[0]` với `qd[0]`: **nhỏ hơn thắng**, rồi `append(chỉ số + n)` cho **người thắng** (không phải người thua)?
- [ ] `popleft()` **cả hai** hàng mỗi lần lặp?
- [ ] Hàng nào còn phần tử thì phe đó thắng (`"Radiant" if qr else "Dire"`)?
- [ ] Không dùng `list.pop(0)` (chậm); dùng `deque`?
- [ ] Đã thử `"RD"`, `"RDD"`, `"R"`, và phản ví dụ "số đông" `"RRDDD"` (`Radiant`)?
- [ ] Skeleton đúng `class Solution` / `def predictPartyVictory(self, senate: str) -> str`?